# ML Pipeline

> Notebook generated from [`examples/subgraphs/01_ml_pipeline.py`](../../examples/subgraphs/01_ml_pipeline.py).

| Item | Value |
|------|-------|
| **Dataset** | UCI Heart Disease (embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** data_ingester → EDA → features → train → eval → [quality gate ≥0.7] → export. Per-stage metrics + exported artifacts.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
ML Pipeline Subgraph — Complete Machine Learning pipeline
=============================================================
Subgraph: prismal.agents.subgraphs.ml_pipeline

Dataset: UCI Heart Disease / Iris / Breast Cancer (scikit-learn datasets)
  • Heart Disease: 303 records, 14 features, binary prediction.
  • Reference: https://archive.ics.uci.edu/ml/datasets/heart+disease
  • Why: The ML Pipeline has 6 specialized nodes (ingestion, EDA,
    feature engineering, training, evaluation, export) with a
    quality gate that requires score >= 0.7. UCI datasets are
    classic benchmarks for demonstrating end-to-end pipelines.

ML Pipeline subgraph description:
  data_ingester → eda_analyst → feature_engineer → model_trainer
  → model_evaluator → [quality gate: score>=0.7?] → model_exporter
  - If score < 0.7 → re-train (max 3 iterations)

Nodes:
  1. data_ingester     — loads and validates CSV/Parquet data
  2. eda_analyst       — exploratory analysis (distributions, correlations)
  3. feature_engineer  — feature selection and transformation
  4. model_trainer     — ML model training
  5. model_evaluator   — evaluation on test set (accuracy, F1, AUC)
  6. [quality_gate]    — if primary_score < 0.7 → retrain (max 3)
  7. model_exporter    — model serialization and export

Usage:
    uv run python examples/subgraphs/01_ml_pipeline.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from prismal.agents.state import initial_state
from prismal.agents.subgraphs.ml_pipeline.builder import (
    build_ml_pipeline_subgraph,
    register_ml_pipeline,
)

## Dataset: problem description and data

In [ ]:
# In a real environment, the agent accesses real CSV/Parquet files.
# Here we provide metadata that the agent will use to orchestrate the pipeline.

ML_TASK = {
    "dataset_name": "UCI Heart Disease",
    "dataset_path": "data/heart_disease.csv",  # path the agent would use
    "description": (
        "Heart disease prediction for 303 patients. "
        "14 features: age, sex, chest pain type, "
        "blood pressure, cholesterol, fasting glucose, "
        "ECG results, maximum heart rate, angina, "
        "ST depression, ST slope, number of colored vessels, "
        "thal, and target (0=no disease, 1=disease)."
    ),
    "target_column": "target",
    "task_type": "binary_classification",
    "evaluation_metric": "accuracy",
    "target_score": 0.80,
    "train_test_split": 0.8,
    "algorithms_to_try": ["RandomForest", "GradientBoosting", "LogisticRegression"],
}


async def main() -> None:
    print("=" * 70)
    print("  ML Pipeline Subgraph — Dataset: UCI Heart Disease")
    print("=" * 70)

    # Register and initialize the subgraph
    print("\n[Initializing ML Pipeline subgraph]")
    await register_ml_pipeline()
    subgraph = await build_ml_pipeline_subgraph()
    print("  ✓ Subgraph compiled with 6 nodes + quality gate")

    # Show architecture
    print("\n[Subgraph architecture]")
    print("  data_ingester")
    print("       ↓")
    print("  eda_analyst")
    print("       ↓")
    print("  feature_engineer")
    print("       ↓")
    print("  model_trainer")
    print("       ↓")
    print("  model_evaluator")
    print("       ↓")
    print("  [quality_gate: score >= 0.7?]")
    print("       ├── YES → model_exporter → END")
    print("       └── NO  → model_trainer (re-training, max 3 cycles)")

    # Prepare the initial state
    state = initial_state()
    state["messages"] = []
    state["metadata"] = {
        "task": ML_TASK,
        "pipeline": "ml_pipeline",
        "thread_id": "ml_heart_disease_001",
    }

    # LangGraph thread configuration
    config = {
        "configurable": {
            "thread_id": "ml_heart_disease_001",
            "recursion_limit": 20,
        }
    }

    # Initial message with instructions
    from langchain_core.messages import HumanMessage

    state["messages"] = [
        HumanMessage(
            content=(
                f"Run a complete Machine Learning pipeline for the dataset: "
                f"{ML_TASK['dataset_name']}.\n\n"
                f"Description: {ML_TASK['description']}\n\n"
                f"Task: {ML_TASK['task_type']}\n"
                f"Target metric: {ML_TASK['evaluation_metric']} >= {ML_TASK['target_score']}\n"
                f"Algorithms to try: {', '.join(ML_TASK['algorithms_to_try'])}"
            )
        )
    ]

    print(f"\n[Running pipeline for: {ML_TASK['dataset_name']}]")
    print(f"  Task         : {ML_TASK['task_type']}")
    print(f"  Minimum score: {ML_TASK['target_score']}")
    print(f"  Algorithms   : {ML_TASK['algorithms_to_try']}")

    # Execute the subgraph
    try:
        final_state = await subgraph.ainvoke(state, config=config)

        print("\n[Pipeline results]")

        # Extract metrics from metadata
        pipeline_meta = final_state.get("metadata", {}).get("ml_pipeline", {})
        if pipeline_meta:
            print(f"  Final score   : {pipeline_meta.get('primary_score', 'N/A'):.3f}")
            print(f"  Chosen model  : {pipeline_meta.get('best_model', 'N/A')}")
            print(f"  Iterations    : {pipeline_meta.get('training_iterations', 1)}")
            print(f"  Features used : {pipeline_meta.get('n_features', 'N/A')}")
            print(f"  Exported model: {pipeline_meta.get('export_path', 'N/A')}")

        # Last message from the agent
        messages = final_state.get("messages", [])
        if messages:
            print("\n  Last pipeline message:")
            print(f"  {str(messages[-1].content)[:300]}")

    except Exception as exc:
        print(f"\n  Execution error: {exc}")
        print("  (Verify the dataset is available and the LLM is configured)")

    # Demonstrate direct use of individual nodes
    print("\n[Alternative usage: individual nodes]")
    print("  from prismal.agents.subgraphs.ml_pipeline import (")
    print("      data_ingester_node,")
    print("      eda_analyst_node,")
    print("      feature_engineer_node,")
    print("      model_trainer_node,")
    print("      model_evaluator_node,")
    print("      model_exporter_node,")
    print("  )")
    print("  # Each node can be used independently in your own graph")

    # Quality gate
    print("\n[Quality Gate]")
    print("  score_gate evaluates metadata['ml_pipeline']['primary_score']")
    print("  If score >= 0.7 → exports the model")
    print("  If score < 0.7  → re-trains (max 3 attempts)")
    print("  If after 3 attempts score still < 0.7 → exports the best model obtained")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()